摘要
-------

**注意** : 本实验需要投入一些时间，请尽早开始工作。

* 截止日期: **2026-04-06   23:59:59**

* 分数: 在实验课程中占 **30 分** 。

<!-- * 迟交惩罚: 每小时扣除 _0.089 分_。迟交七天本实验分数登记为 _0 分_。 -->

实验说明
------------

- 该文本包含 **实验 1** 的实验说明。

- 需要在 `vector.c` 和 `vector.source`文件中进行代码实现。

- 不允许在屏幕或日志文件中打印不需要的信息。考核中不会考虑类似的输出。

<!-- - 需要将你的实现提交到对应链接: https://szudseg.cn/submit/ -->

- 我们提供每一部分的详细说明，如果有更多的问题，可以在微信群中进行提问。

<!-- - 仅允许使用标准库的函数，不允许使用第三方库。 -->

- 对提交的文件我们会 **快速反馈** ，你可以在线上提交平台阅读反馈信息。

- 允许 **不限次数** 的提交, 我们会使用 **最后** 提交的文件进行评估。 

- 请使用 **Postgres 12.5** 完成本实验。

实验 1：在 PostgreSQL 添加 vector 数据类型 
----------

实验目标：

*   理解DBMS如何存储以及处理数据。
*   练习在 PostgreSQL 中添加新的基础数据类型。
<!-- *   了解在 PostgreSQL 中使用对新的结构定义索引结构。 -->

实现新的数据类型，并完成 input/output 函数与一系列的 operations。

请在开始工作前 _认真阅读全部_ 实验说明。没有阅读实验说明而导致的问题都会回复“请阅读实验说明”。

在说明中我们会使用下述名词指代：

`PG_CODE`: PostgreSQL 源码所在目录  (例如 `/srvr/$USER/postgresql-12.5`)

`PG_HOME`: PostgreSQL 二进制文件安装目录  (例如 `/srvr/$USER/pgsql`)

`PG_DATA`: 存放 PostgreSQL's `data` 的目录(例如 `/srvr/$USER/pgsql/data`)

`PG_LOG`: PostgreSQL 日志输出文件 (例如 `/srvr/$USER/pgsql/log`)

引言
------------

PostgreSQL数据库系统具有良好的可扩展性，我们能够将新数据类型添加到 PostgreSQL 数据库系统中，这中可扩展性允许 PostgreSQL 用户开发多种数据类型（例如多边形），且这些数据类型已成为标准版本的一部分，在这个作业中，我们将添加一个新的数据类型来处理 **vector向量**。

PostgreSQL 文档的以下部分描述了在 PostgreSQL 中添加新的基本数据类型的过程：
*   [37.10 C-Language Functions](https://www.postgresql.org/docs/12//xfunc-c.html)
*   [37.13 User-defined Types](https://www.postgresql.org/docs/12//xtypes.html)
*   [37.14 User-defined Operators](https://www.postgresql.org/docs/12//xoper.html)
*   [SQL: CREATE TYPE](https://www.postgresql.org/docs/12//sql-createtype.html)
*   [SQL: CREATE OPERATOR](https://www.postgresql.org/docs/12//sql-createoperator.html)
*   [SQL: CREATE OPERATOR CLASS](https://www.postgresql.org/docs/12//sql-createopclass.html)

设置
----------

你应该从PostgreSQL 12.5 开始完成这个实验，你只需为此配置、编译和安装一次 PostgreSQL 服务器。所有后续编译都在 `src/tutorial` 目录中进行，并且只需要修改那里的文件。

一旦重新安装了 PostgreSQL 服务器，你应该运行以下命令：
``` console
cd PG\_CODE/src/tutorial
cp complex.c vector.c
cp complex.source vector.source
```

一旦你制作了 `vector` 文件，你还应该编辑这个目录中的 `Makefile`，并将文本添加到以下行：
``` console
MODULES = complex funcs vector
DATA_built = advanced.sql basics.sql complex.sql funcs.sql syscat.sql vector.sql
```

该作业的其余工作涉及编辑 `vector.c` 和 `vector.source` 文件，为了使 `Makefile` 正常工作，你必须使用 `vector.source` 文件中的标识符 `_OBJWD_` 来引用保存已编译库的目录，不要直接修改由 `Makefile` 生成的 `vector.sql` 文件，同时通过以下命令导入对应的数据类型、函数以及操作符号。
```sql
\i vector.sql
```

<!-- 你可以使用其他 `*.c` 文件和 `vector.c`，但你需要对 `Makefile` 做进一步的修改，以确保它们被编译并正确链接到编译库。 -->

vector 数据类型
--------------------

我们的目标是定义一个新的基本类型`vector`，来存储浮点数向量的概念，还打算在 `vector` 类型上定义一些运算。

我们将向量表示为花括号、逗号分隔的浮点数集合：'\{1.0, -2.1, 3.4, 5.0, 5.1\}'。

### 实现 vector

你需要做的第一件事是定义 `vector` 这个数据类型，鉴于 `vector` 可以存储任意维度的向量，所以**不能**使用拥有固定大小对象来保存 `vector` 类型值，

你可以参考Complex与VarChar来自定义 `vector` 数据类型。 请注意，Complex类型是一个简单的例子，主要用来了解如何添加新数据类型。不要误以为这个实验只需要将名称 complex 更改为 vector；vector 类型比复数类型更复杂, 在 tutorial 和 contrib 目录下还有其他新数据类型的示例以供学习以及参考。
*   Complex datatype  
    `PG_CODE/src/tutorial/complex.c` 与 `PG_CODE/src/tutorial/complex.sql`
*   VarChar datatype  
    `PG_CODE/src/include/c.h`(565行，数据结构定义) 与 `PG_CODE/src/backend/adt/varlena.c` (182行，数据结构初始化)

当你读取表示 `vector` 值的字符串时，它们会被转换成内部表示的`vector`数据结构，以这种表示存储在数据库中，并使用这种数据结构对 `vector` 值进行操作。当你打印 `vector` 值时，你应该以规范的形式打印它们，无论它们是如何输入或如何存储的，输入和输出的范式如后文给出的例子。

首先，你需要编写函数是输入和输出类型为 `vector` 的值的函数。你应该编写与文件 `complex.c` 中定义的函数 `complex_in()` 和 `complex_out()` 的类似函数。函数名称应是 `vector_in()` 和 `vector_out()`。 确保使用 `V1` 风格的函数接口（就像 `complex.c` 中所做的那样）。请注意，两个 输入/输出 函数应该是互补的，这意味着输出函数显示的任何字符串都必须能够使用输入函数读取。你不需要保留用于输入的精确字符串（例如，你可以在内部以规范形式存储 `vector` 值）

你 _不需要_ 在 PostgreSQL 文档中定义称为 `receive_function` 和 `send_function` 的二进制输入/输出函数。二进制输入/输出函数在 `complex.c` 文件中称为 `complex_send()` 和 `complex_recv()`

**注意事项：** 
* `vector`的每一维度都是**float**类型（请不要用double，且在后续的测试中，我们也保证每一维度都在float可以表示的范围内，而且在后续的测试中每一维度我们都保证最多到小数点后6位），同时我们保证vector最多含有1024维度。
* 你可以使用 **strtok** 函数来完成字符串的分割，以及采用**strtof**来将字符串转换成为float类型或者报错，请自行学习API的接口细节。
* 你**可以使用float_to_shortest_decimal_bufn、float_to_shortest_decimal等类型转换功能函数**来将float类型转换成为字符串类型，同时包含对应头文件`#include "common/shortest_dec.h"`，这些函数的原型在`PG_CODE/src/common/f2s.c`，因为这样才能保证输出的一致性，'{1.1,2.3,4}'同时输出的字符串应该以','分割，同时**不**应该包含任何多余的空格。
* 在你尝试在 PostgreSQL 中测试之前，请在 PostgreSQL 之外尽可能多地测试你的 C 函数（例如，编写一个简单的测试程序）,这将使调试更容易。

### vector运算

你必须为 `vector` 类型实现以下的操作。(假设 A、B 和 S 是 `vector`)：

`<#> S` ：返回`vector S`的维度（应该为integer类型）

`A <-> B`：返回`vectorA` 与 `vector B`的之间的L2距离（应该为float4类型，不同维度之间的vector不能计算L2距离）

**注意事项：在写vector.source文件的时候一定注意返回值一定是float4， float类型在SQL语言中其实是8个字节，对应C语言中的double类型**

`A - B`：返回`vector A` 与 `vector B`每一维度相减得到新的vector（不同维度之间的vector不能相加）。

`A + B`：返回`vector A` 与 `vector B`每一维度相加得到新的vector（不同维度之间的vector不能相加）。

### vector示例
``` sql 

create table test_vector (id integer primary key, vec vector);
CREATE TABLE

insert into test_vector values (1, '{-1.2, 2.1345, 3}');
INSERT 0 1



select <#>'{1,2}';
 ?column? 
----------
        2
(1 row)

select '{1,2}'::vector<->'{3,4}'::vector;
 ?column? 
----------
 2.828427
(1 row)


select *, '{1,2,3}'<->vec as dis from test_vector order by dis  limit 10;
 id |        v        |    dis    
----+-----------------+-----------
  1 | {-1.2,2.1345,3} | 2.2041075
(1 row)

-- 显然由于float引发了一定的误差，所以在输出vector的时候请一定用float_to_shortest_decimal_bufn函数
select '{1.1,2.2}'::vector-'{3.3,4.4}'::vector;
     ?column?      
-------------------
 {-2.1999998,-2.2}
(1 row)

-- etc. etc. etc.

select '{1.1,2.2}'::vector+'{3.3,4.4}'::vector;
    ?column?     
-----------------
 {4.4,6.6000004}
(1 row)


valid vector:
'{+0}'
'{1}'
'{-0.0}'
'{1, 2, 3}'
'{1, 2.1, 3}'
'{-2.1, 3, 1}'
'{+2.1, 3, 1}'
'{-1.111111}'
'{  1, 2,3}'
'{6,6.0,6.1,6,6,6}'
'{1, 01, 001, 0001}'


invalid vector:
'{}'  (vector 至少为1维)
'{,}'
'{1,}'
'{1, 2, 3} '
' {1, 2, 3}'(vector 必须以'{'开头以及以'}'结尾)
'{++1.0, 2,3}'
'{--1.0, 2,3}'
'{-1. 0, 2,3}'
'{+1.a, 2,3}'
'{-1 23, 2,3}'
'{- 123, 2,3}'
'{1, {2,3}, 4}'
'{{1,2,3,5}'
'{7,17,,27,37}'
'{-123 , 2,3}'(在输入的每一维度的字符串中在中间以及后面不存在额外的空格，但是在前面可以存在空格，请一定使用strtof函数处理每一维度)

```

提交
----------

你需要提交两个文件： 

`vector.c` - 包含实现了 `vector` 数据类型的C语言函数。 

`vector.source` - 包含安装 `vector` 数据类型到 PostgreSQL 服务器的SQL模板。

请 _不要_ 提交 `vector.sql` 文件，因为该文件包含文件的绝对路径，不会用在我们的测试环境中。

请 _不要_ 在文件 `vector.source` 包括:

*   `create table ...`
*   `insert into ...`
*   `select ...`
*   `drop type ...`

或者其他创建 `vector` 数据类型中不需要的语句。


评分标准
---------

本实验的评分标准如下：

- 输入/输出 
    <!-- - 有效: 25 (处理有效输入) -->
    <!-- - 无效: 5 (识别错误或无效输入并反馈) -->
    - input_basic: 20
    - input_tricky: 20
    - output: 20
- 操作符
    - vector_dim(<#>): 10
    - l2_distance(<->): 10
    - vector_add(+): 10
    - vector_sub(-): 10
- (总分: **100**)

**注意**：在反馈系统中，该实验满分为`100`，在课程总分评估中会按比例缩小。